# Dispatch Region Sum Source

In [ ]:
from pathlib import Path
import traceback
import nemosis
import pandas as pd




Fetch and cache dispatch region summary metrics from NEMOSIS.

In [ ]:

def _process_month(start: str, end: str) -> pd.DataFrame:
    REGION_SUFFIX = {"NSW1": "_nsw", "QLD1": "_qld", "VIC1": "_vic", "SA1": "_sa"}
    RAW_FIELDS    = ["TOTALDEMAND", "AVAILABLEGENERATION", "NETINTERCHANGE", "DEMANDFORECAST", "DISPATCHABLEGENERATION"]
    OUTPUT_FIELDS = ["demand", "avail_gen", "interchange", "demand_forecast", "dispatch_gen"]

    df = nemosis.dynamic_data_compiler(
        start_time=start,
        end_time=end,
        table_name="DISPATCHREGIONSUM",
        raw_data_location=str(Path("Pre_processing/nemosis_raw_dispatch_region_sum")),
        select_columns=["SETTLEMENTDATE", "REGIONID"] + RAW_FIELDS,
        filter_cols=["REGIONID"],
        filter_values=[list(REGION_SUFFIX.keys())],
        fformat="feather",
        keep_csv=False,
    )
    df["SETTLEMENTDATE"] = pd.to_datetime(df["SETTLEMENTDATE"])
    for col in RAW_FIELDS:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    region_frames = []
    for region, suffix in REGION_SUFFIX.items():
        rdf = df[df["REGIONID"] == region][["SETTLEMENTDATE"] + RAW_FIELDS].copy()
        rdf = (
            rdf.set_index("SETTLEMENTDATE")
            .sort_index()
            .resample("5min")
            .mean(numeric_only=True)
        )
        rdf.columns = [f"{name}{suffix}" for name in OUTPUT_FIELDS]
        region_frames.append(rdf)
    return pd.concat(region_frames, axis=1).sort_index()


def main():
    data_start = pd.Timestamp("2018/01/01")
    data_end   = pd.Timestamp("2026/01/01")

    processed_path = Path("Processed_data/2_dispatch_region_sum.csv")
    already_processed = set()
    if processed_path.exists():
        existing_idx = pd.read_csv(processed_path, usecols=[0], parse_dates=True).iloc[:, 0]
        already_processed = set(pd.to_datetime(existing_idx).dt.to_period("M"))
        print(f"  Found {len(already_processed)} already-processed month(s) — will skip.", flush=True)

    total_months = pd.date_range(data_start, data_end - pd.Timedelta(days=1), freq="MS")
    monthly_dfs = []

    for i, month in enumerate(total_months):
        if month.to_period("M") in already_processed:
            print(f"{i + 1:3d}/{len(total_months)} {month:%Y-%m} skipping (already processed).", flush=True)
            continue
        start = month.strftime("%Y/%m/%d %H:%M:%S")
        end   = (month + pd.offsets.MonthEnd(0) + pd.Timedelta(days=1)).strftime("%Y/%m/%d %H:%M:%S")
        print(f"{i + 1:3d}/{len(total_months)} {month:%Y-%m} fetching...", flush=True)
        try:
            monthly_dfs.append(_process_month(start, end))
            for f in Path("Pre_processing/nemosis_raw_dispatch_region_sum").glob(f"*{month.strftime('%Y%m')}*.feather"):
                f.unlink()
            print(f"  {month:%Y-%m} raw files deleted.", flush=True)
        except Exception:
            print(f"  WARNING: {month:%Y-%m} failed:\n{traceback.format_exc()}", flush=True)

    if not monthly_dfs:
        print("No new months to process.", flush=True)
        return pd.read_csv(processed_path, index_col=0, parse_dates=True).tail()

    df = pd.concat(monthly_dfs).sort_index()
    df = df[~df.index.duplicated(keep="last")]

    if processed_path.exists():
        existing = pd.read_csv(processed_path, index_col=0, parse_dates=True)
        existing.index = pd.to_datetime(existing.index, format="mixed")
        df = pd.concat([existing, df])
        df = df[~df.index.duplicated(keep="last")].sort_index()

    df.index.name = "SETTLEMENTDATE"
    df.to_csv(processed_path)
    return df.tail()

main()


  Found 97 already-processed month(s) — will skip.
  1/96 2018-01 skipping (already processed).
  2/96 2018-02 skipping (already processed).
  3/96 2018-03 skipping (already processed).
  4/96 2018-04 skipping (already processed).
  5/96 2018-05 skipping (already processed).
  6/96 2018-06 skipping (already processed).
  7/96 2018-07 skipping (already processed).
  8/96 2018-08 skipping (already processed).
  9/96 2018-09 skipping (already processed).
 10/96 2018-10 skipping (already processed).
 11/96 2018-11 skipping (already processed).
 12/96 2018-12 skipping (already processed).
 13/96 2019-01 skipping (already processed).
 14/96 2019-02 skipping (already processed).
 15/96 2019-03 skipping (already processed).
 16/96 2019-04 skipping (already processed).
 17/96 2019-05 skipping (already processed).
 18/96 2019-06 skipping (already processed).
 19/96 2019-07 skipping (already processed).
 20/96 2019-08 skipping (already processed).
 21/96 2019-09 skipping (already processed).
 22/

,demand_nsw,avail_gen_nsw,interchange_nsw,demand_forecast_nsw,dispatch_gen_nsw,demand_qld,avail_gen_qld,interchange_qld,demand_forecast_qld,dispatch_gen_qld,demand_vic,avail_gen_vic,interchange_vic,demand_forecast_vic,dispatch_gen_vic,demand_sa,avail_gen_sa,interchange_sa,demand_forecast_sa,dispatch_gen_sa
SETTLEMENTDATE,,,,,,,,,,,,,,,,,,,,
2025-12-31 23:40:00,6815.60,12485.12732,-600.06,-23.0,6215.54,6510.48,10028.36101,-420.97,-15.0,6096.50,4242.66,10074.71718,628.24,-30.0,4870.90,1470.70,3906.46992,605.86,26.0,2076.55
2025-12-31 23:45:00,6747.40,12495.33687,-523.70,-33.0,6226.70,6525.66,9907.03286,-501.20,-2.0,6027.46,4218.78,10042.68777,628.90,-35.0,4847.68,1451.36,3926.24992,623.97,4.0,2075.33
2025-12-31 23:50:00,6787.87,12487.70049,-567.26,-21.0,6220.60,6502.55,9815.94862,-435.07,-8.0,6088.47,4201.52,10018.41088,632.91,-35.0,4834.43,1445.97,3900.62022,579.72,0.0,2025.68
2025-12-31 23:55:00,6751.40,12512.82807,-506.24,-31.0,6245.17,6450.74,9763.03977,-541.79,-29.0,5925.95,4162.44,9993.67413,619.86,-36.0,4782.30,1415.50,3902.42120,676.50,-2.0,2092.00
2026-01-01 00:00:00,6777.74,12504.21847,-540.13,-24.0,6237.61,6424.81,9755.17063,-401.79,-38.0,6024.02,4136.51,9814.17370,589.91,-32.0,4726.42,1437.95,3851.48796,525.33,4.0,1963.29
